# Calçada Viva — IA Generativa para Engenharia

**Desafio CitizenTech | Engenharia de Software — FIAP**

Este notebook demonstra um assistente que organiza relatos sobre barreiras de acessibilidade em calçadas. O cidadão descreve o que encontrou; o sistema classifica o relato, resume os fatos e pergunta o que falta. A pessoa revisa o resultado antes de compartilhá-lo.

> **Modo de execução:** por padrão, o notebook roda em modo demonstrativo e produz saídas visíveis sem chave ou custo de API. Para chamar o modelo de verdade, configure `OPENROUTER_API_KEY` no ambiente e `CALCADA_VIVA_MODE=api` antes de executar. As saídas do modo demonstrativo são exemplos fixos, não respostas geradas pelo modelo.

**Integrantes:** Gustavo Felex (RM554242), Vinicius Issa Gois (RM553814), Gustavo Bonani (RM553493), Wesley Vieira (RM553496).

## Modificação do aluno 1 — seleção do modelo via API

O modelo foi configurado para **`openai/gpt-4o-mini`** pelo OpenRouter. A escolha prioriza uma chamada simples por API compatível com Chat Completions e um modelo leve para classificação e organização de textos curtos. A configuração fica centralizada em uma variável para facilitar a troca e comparação com o modelo usado em aula.

O notebook foi preparado como uma base autônoma porque o arquivo original usado em aula não foi incluído nos materiais recebidos. Se a disciplina exigir a edição de um notebook específico do canal compartilhado, as células identificadas como modificações do aluno podem ser copiadas para esse arquivo.

A API usa o endpoint documentado pelo OpenRouter: `https://openrouter.ai/api/v1/chat/completions`. A chave fica fora do notebook, em variável de ambiente.

In [1]:
# MODIFICACAO-ALUNO
import json
import os
import urllib.request
import urllib.error

BASE_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_ID = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
API_KEY = os.getenv("OPENROUTER_API_KEY")
MODO = os.getenv("CALCADA_VIVA_MODE", "demo").strip().lower()

if MODO not in {"demo", "api"}:
    raise ValueError("CALCADA_VIVA_MODE deve ser 'demo' ou 'api'.")

print(f"Modelo configurado: {MODEL_ID}")
print(f"Modo: {MODO}")
if MODO == "api" and not API_KEY:
    raise RuntimeError("Defina OPENROUTER_API_KEY no ambiente antes de usar o modo api.")


def chamar_api(mensagens, max_tokens=250):
    """Envia uma conversa ao endpoint Chat Completions do OpenRouter."""
    payload = json.dumps({
        "model": MODEL_ID,
        "messages": mensagens,
        "temperature": 0.2,
        "max_tokens": max_tokens,
    }).encode("utf-8")
    request = urllib.request.Request(
        BASE_URL,
        data=payload,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://fiap.com.br/",
            "X-Title": "Calçada Viva - FIAP",
        },
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        data = json.loads(response.read().decode("utf-8"))
    return data["choices"][0]["message"]["content"]


def gerar_resposta(mensagens, resposta_demo, max_tokens=250):
    """Usa API real quando configurado; senão, devolve fixture demonstrativa."""
    if MODO == "demo":
        return resposta_demo
    return chamar_api(mensagens, max_tokens=max_tokens)


def parsear_json(texto):
    """Valida JSON simples sem tentar adivinhar ou reparar resposta inválida."""
    try:
        valor = json.loads(texto)
        if not isinstance(valor, dict):
            raise ValueError("A resposta JSON precisa ser um objeto.")
        return valor, None
    except (json.JSONDecodeError, ValueError) as erro:
        return None, str(erro)

print("Funções de chamada, modo demonstrativo e validação carregadas.")

Modelo configurado: openai/gpt-4o-mini
Modo: demo
Funções de chamada, modo demonstrativo e validação carregadas.


## Modificação do aluno 2 — prompt para classificar e resumir

**Justificativa:** relatos livres variam bastante. Um conjunto fechado de categorias e a instrução para não inventar fatos tornam a resposta mais consistente e segura. O prompt também pede que a IA indique quando não consegue classificar.

In [2]:
# MODIFICACAO-ALUNO
relato_1 = (
    "Tem caixas ocupando a calçada na Rua das Flores, perto do número 120. "
    "Uma pessoa de cadeira de rodas não consegue passar."
)

mensagens_1 = [
    {
        "role": "system",
        "content": (
            "Você organiza relatos sobre acessibilidade em calçadas. "
            "Classifique em: buraco, desnível, bloqueio, falta de rampa, "
            "piso danificado, outro ou insuficiente. Resuma somente fatos informados. "
            "Não invente endereço, causa ou responsável. Se faltar um dado essencial, "
            "indique qual informação deve ser perguntada. Responda em português simples."
        ),
    },
    {"role": "user", "content": relato_1},
]

resposta_1 = gerar_resposta(
    mensagens_1,
    "Categoria: bloqueio. Resumo: caixas ocupam a calçada e impedem a passagem de uma pessoa em cadeira de rodas. Informação que falta: bairro ou ponto de referência próximo.",
)
print("Relato:", relato_1)
print("Resposta:", resposta_1)

Relato: Tem caixas ocupando a calçada na Rua das Flores, perto do número 120. Uma pessoa de cadeira de rodas não consegue passar.
Resposta: Categoria: bloqueio. Resumo: caixas ocupam a calçada e impedem a passagem de uma pessoa em cadeira de rodas. Informação que falta: bairro ou ponto de referência próximo.


## Modificação do aluno 3 — prompt com saída estruturada em JSON

**Justificativa:** campos previsíveis facilitam uma futura integração com um formulário. A resposta mantém localização ausente como `null`, explicita dados pendentes e sinaliza revisão humana. O resultado não é tratado como vistoria oficial.

In [3]:
# MODIFICACAO-ALUNO
mensagens_2 = [
    {
        "role": "system",
        "content": (
            "Extraia do relato apenas informações explícitas e devolva somente JSON válido, "
            "sem bloco markdown. Use exatamente as chaves: categoria, descricao_resumida, "
            "local, impacto_acessibilidade, informacoes_pendentes, requer_revisao_humana. "
            "Categorias permitidas: buraco, desnível, bloqueio, falta de rampa, "
            "piso danificado, outro, insuficiente. Impacto deve ser baixo, médio, alto "
            "ou não avaliado; é uma triagem preliminar, não um laudo. Use null se o local "
            "não foi informado e true para requer_revisao_humana. Não invente dados."
        ),
    },
    {"role": "user", "content": relato_1},
]

json_demo = json.dumps({
    "categoria": "bloqueio",
    "descricao_resumida": "Caixas ocupam a calçada e impedem a passagem de uma pessoa em cadeira de rodas.",
    "local": "Rua das Flores, perto do número 120",
    "impacto_acessibilidade": "alto",
    "informacoes_pendentes": ["bairro ou ponto de referência próximo"],
    "requer_revisao_humana": True,
}, ensure_ascii=False)

texto_json = gerar_resposta(mensagens_2, json_demo, max_tokens=300)
dados, erro_json = parsear_json(texto_json)
if erro_json:
    print("A resposta não veio em JSON válido; solicitar nova tentativa ou revisão humana.")
    print("Detalhe:", erro_json)
else:
    print(json.dumps(dados, ensure_ascii=False, indent=2))

{
  "categoria": "bloqueio",
  "descricao_resumida": "Caixas ocupam a calçada e impedem a passagem de uma pessoa em cadeira de rodas.",
  "local": "Rua das Flores, perto do número 120",
  "impacto_acessibilidade": "alto",
  "informacoes_pendentes": [
    "bairro ou ponto de referência próximo"
  ],
  "requer_revisao_humana": true
}


## Modificação do aluno 4 — prompt para informação ausente e resposta acessível

**Justificativa:** localização incompleta não deve levar a uma resposta inventada. O assistente pede um ponto de referência de forma simples, evita solicitar dados pessoais e não promete que o relato foi enviado a um órgão público.

In [4]:
# MODIFICACAO-ALUNO
relato_3 = "Tem um negócio atrapalhando a calçada aqui e está difícil passar."
mensagens_3 = [
    {
        "role": "system",
        "content": (
            "Responda em português simples, com frases curtas e tom respeitoso. "
            "Não invente localização nem tipo de obstáculo. Se faltar informação, "
            "faça uma pergunta objetiva e peça bairro, endereço aproximado ou ponto de "
            "referência. Não peça nome, telefone ou CPF. Não diga que a prefeitura recebeu "
            "ou resolveu o relato."
        ),
    },
    {"role": "user", "content": relato_3},
]
resposta_3 = gerar_resposta(
    mensagens_3,
    "Você pode dizer o que está bloqueando a passagem e informar o bairro ou um ponto de referência próximo? Não precisa enviar seus dados pessoais.",
)
print("Relato:", relato_3)
print("Pergunta de esclarecimento:", resposta_3)

Relato: Tem um negócio atrapalhando a calçada aqui e está difícil passar.
Pergunta de esclarecimento: Você pode dizer o que está bloqueando a passagem e informar o bairro ou um ponto de referência próximo? Não precisa enviar seus dados pessoais.


## Experimento complementar — tratamento de erros do modelo

**Justificativa:** modelos de linguagem às vezes devolvem texto fora do formato pedido (JSON cortado, bloco markdown, lista em vez de objeto, texto livre). Este experimento passa respostas inválidas **reais** pela função `parsear_json()` e verifica que cada uma é detectada, em vez de ser aceita ou "consertada" por adivinhação. Também mostra a mensagem segura exibida ao cidadão quando a chamada à API falha (tempo limite simulado).

In [5]:
# MODIFICACAO-ALUNO
# Teste 1: respostas que um modelo pode devolver fora do formato pedido
respostas_invalidas = {
    "JSON cortado no meio": '{"categoria": "bloqueio", "local": "Rua das Flo',
    "JSON dentro de bloco markdown": '```json\n{"categoria": "buraco"}\n```',
    "lista em vez de objeto": '["bloqueio", "Rua das Flores"]',
    "texto livre": "Categoria: bloqueio. Local: Rua das Flores.",
}

print("Teste 1 — validação de respostas inválidas do modelo")
todas_detectadas = True
for descricao, texto in respostas_invalidas.items():
    dados_teste, erro = parsear_json(texto)
    detectado = dados_teste is None and erro is not None
    todas_detectadas = todas_detectadas and detectado
    print(f"  [{'OK' if detectado else 'FALHOU'}] {descricao}: {erro}")

valido, erro_valido = parsear_json('{"categoria": "buraco", "local": null}')
print(f"  [{'OK' if valido and not erro_valido else 'FALHOU'}] JSON válido aceito: {valido}")
assert todas_detectadas and valido is not None
print("Resultado: todas as respostas inválidas foram detectadas.\n")


# Teste 2: falha na chamada à API (tempo limite simulado)
def resposta_de_fallback(erro):
    return (
        "Não consegui organizar o relato agora. Tente novamente mais tarde "
        "ou revise as informações manualmente. Nenhum relato foi enviado."
    )

print("Teste 2 — falha na chamada à API")
try:
    raise TimeoutError("simulação controlada de tempo limite da API")
except (TimeoutError, urllib.error.URLError) as erro:
    mensagem_segura = resposta_de_fallback(erro)
    print("  Falha simulada:", type(erro).__name__)
    print("  Mensagem ao cidadão:", mensagem_segura)

Teste 1 — validação de respostas inválidas do modelo
  [OK] JSON cortado no meio: Unterminated string starting at: line 1 column 36 (char 35)
  [OK] JSON dentro de bloco markdown: Expecting value: line 1 column 1 (char 0)
  [OK] lista em vez de objeto: A resposta JSON precisa ser um objeto.
  [OK] texto livre: Expecting value: line 1 column 1 (char 0)
  [OK] JSON válido aceito: {'categoria': 'buraco', 'local': None}
Resultado: todas as respostas inválidas foram detectadas.

Teste 2 — falha na chamada à API
  Falha simulada: TimeoutError
  Mensagem ao cidadão: Não consegui organizar o relato agora. Tente novamente mais tarde ou revise as informações manualmente. Nenhum relato foi enviado.


## Como ativar a chamada real ao modelo

1. Instale o ambiente indicado em `requirements.txt` e abra este arquivo em Jupyter ou Google Colab.
2. Crie uma chave no OpenRouter e configure-a como variável de ambiente `OPENROUTER_API_KEY`. **Não cole a chave em células e não a envie para o GitHub.**
3. Configure `CALCADA_VIVA_MODE=api`. Se quiser trocar de modelo, defina `OPENROUTER_MODEL` com o identificador escolhido e registre a justificativa.
4. Reinicie o kernel e execute todas as células. Chamadas reais podem gerar cobrança conforme o modelo e a conta.
5. Salve uma cópia executada com as saídas reais e faça os prints para a entrega. O notebook fornecido já contém saídas visíveis do modo demonstrativo.

### Limitações e ética

- A classificação é preliminar; não é laudo nem confirmação de protocolo.
- A IA pode resumir ou classificar incorretamente. A pessoa deve revisar antes de compartilhar.
- Não envie relatos com nomes, telefones, documentos ou outros dados pessoais.
- A resposta depende do modelo, da API e da conexão; em falha, o sistema apresenta uma mensagem de contingência.
- O protótipo não envia solicitações à prefeitura e não substitui canais oficiais ou serviços de emergência.